# 113 — CPTAC-3 external cohort construction

**Scope:** outcome-blind Phase 1 metadata cohort construction; consumes executed Notebook 111 and frozen TCGA 203; does not score programs or perform inference. The approved **G2** universe is 1,082 simple primary-RNA cases. **G3 protocol is draft**; no patient may be certified independent without reviewed evidence. **No cohort freeze or Phase 8 authorization** is automatic.

**Execution:** Run All from project environment with 111 CSVs in `data/interim/metadata/`. Optional reviewed evidence and lineage crosswalk are manually supplied via documented CSVs; no live API calls. If unavailable, retain unresolved status. Outputs are descriptive/interim under `data/interim/metadata/`, never silently registered.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re
import pandas as pd
from IPython.display import display
from pancancer_epigenetics.utils.paths import Paths
from pancancer_epigenetics.utils.file_checks import calculate_sha256

ROOT = Paths.root
META = Paths.metadata
OUT = META
OUT.mkdir(parents=True, exist_ok=True)
REGISTRY = ROOT / "config" / "artifact_registry.json"
INDEX = META / "111_cptac3_file_index.csv"
PROVISIONAL = META / "111_cptac3_provisional_primary_tumor_files.csv"
G3_EVIDENCE = META / "113_reviewed_donor_evidence.csv"  # optional external, case-level reviewed evidence
G4_CROSSWALK = META / "113_reviewed_source_lineage_crosswalk.csv"  # optional reviewed source-stratum crosswalk
QC_EVIDENCE = META / "113_reviewed_qc_evidence.csv"  # optional per-file QC evaluation, source-method documented

def need(path):
    if not path.is_file(): raise FileNotFoundError(f"Required audited input missing: {path}")
    return path
def load(path):
    return pd.read_csv(need(path), dtype="string").fillna(pd.NA)
def exact_unique(df, keys, label):
    if df.duplicated(keys).any(): raise ValueError(f"{label}: duplicate keys {keys}")
def clean_id(x):
    return x.astype("string").str.strip().str.casefold()
def sha(path):
    return calculate_sha256(path)

with need(REGISTRY).open(encoding="utf-8") as f: registry=json.load(f)
tcga_ref=registry["artifacts"]["phase2.203.final_sample_mapping"]
assert tcga_ref["status"]=="frozen" and tcga_ref["shape"][0]==9965
TCGA=need(ROOT / tcga_ref["path"])
assert sha(TCGA)==tcga_ref["sha256"], "Frozen TCGA mapping SHA-256 mismatch"
source=load(INDEX)
primary=load(PROVISIONAL)
tcga=load(TCGA)
print("Audited inputs:",len(source),"file index rows;",len(primary),"provisional primary files;",len(tcga),"frozen TCGA cases")

Audited inputs: 5578 file index rows; 3181 provisional primary files; 9965 frozen TCGA cases


In [2]:
required={"modality","file_id","case_uuid","case_submitter_id","sample_uuid","aliquot_uuid","mapping_status","sample_type","primary_site","disease_type","project_id","provider_md5"}
assert required.issubset(source.columns), required-set(source.columns)
assert required.issubset(primary.columns), required-set(primary.columns)
exact_unique(source,["modality","file_id"],"111 source")
exact_unique(primary,["modality","file_id"],"111 provisional primary")
assert len(source.loc[source.modality.eq("rnaseq")])==2968
assert len(source.loc[source.modality.eq("methylation")])==2610
rna=source.loc[source.modality.eq("rnaseq")].copy()
candidate=primary.loc[primary.modality.eq("rnaseq")].copy()
assert len(candidate)==1731 and candidate.case_uuid.nunique()==1370, "111 G2 source universe changed"
assert candidate.file_id.isin(rna.file_id).all()
assert candidate.mapping_status.eq("unique_aliquot_sample").all()
assert candidate.sample_type.eq("Primary Tumor").all()
assert candidate[["file_id","case_uuid","sample_uuid","aliquot_uuid"]].notna().all().all()
case_count=candidate.groupby("case_uuid").agg(n_files=("file_id","nunique"),n_samples=("sample_uuid","nunique"),n_aliquots=("aliquot_uuid","nunique"))
simple_cases=case_count.loc[(case_count.n_files==1)&(case_count.n_samples==1)&(case_count.n_aliquots==1)].index
assert len(simple_cases)==1082 and len(case_count)==1370
assert len(case_count)-len(simple_cases)==288
ledger=candidate.loc[candidate.case_uuid.isin(simple_cases)].copy().sort_values("case_uuid").reset_index(drop=True)
assert len(ledger)==1082 and ledger.case_uuid.is_unique
ambig=rna.loc[rna.mapping_status.eq("ambiguous_sample_ancestry")]
assert len(ambig)==314
flagged=set(ambig.case_uuid.dropna()) & set(ledger.case_uuid)
assert len(flagged)==37, f"Additional ambiguous-file flags changed: {len(flagged)}"
ledger["other_ambiguous_rna_flag"]=ledger.case_uuid.isin(flagged)
ledger["g2_status"]="SIMPLE_PRIMARY_CANDIDATE"
mult=case_count.loc[~case_count.index.isin(simple_cases)]
assert len(mult.loc[mult.n_samples>1])==276 and len(mult.loc[(mult.n_samples==1)&(mult.n_aliquots>1)])==12
assert mult.n_files.sum()==649
print("G2 exact:",len(ledger),"simple cases;",len(mult),"multiplicity cases;",len(flagged),"flagged; no biological data used")

G2 exact: 1082 simple cases; 288 multiplicity cases; 37 flagged; no biological data used


In [3]:
## Identity namespace screening is descriptive; no automatic donor equivalence from UUID strings.
for c in ["rna_case_uuid","case_submitter_id"]:
    if c not in tcga.columns: raise ValueError(f"TCGA frozen mapping lacks expected field {c}")
for c in ["case_uuid","case_submitter_id"]: 
    if ledger[c].isna().any(): raise ValueError(f"CPTAC G2 missing {c}")
# A single common string can flag a collision only; donor linkage requires separate evidence.
tcga_uuids=set(clean_id(tcga["rna_case_uuid"]).dropna())
tcga_submitters=set(clean_id(tcga["case_submitter_id"]).dropna())
ledger["uuid_collision_flag"]=clean_id(ledger.case_uuid).isin(tcga_uuids)
ledger["submitter_collision_flag"]=clean_id(ledger.case_submitter_id).isin(tcga_submitters)
ledger["g3_class"]="UNRESOLVED_INDEPENDENCE"
ledger["g3_evidence_id"]=pd.NA
ledger["g3_reason"]="No reviewed, applicable donor-nonoverlap/overlap evidence"
ledger["g3_reviewed"]=False
print("Lexical collision flags (NOT donor-level evidence):",int(ledger.uuid_collision_flag.sum()),int(ledger.submitter_collision_flag.sum()))

Lexical collision flags (NOT donor-level evidence): 0 0


## G3 — reviewed donor evidence (optional)

To apply evidence, provide `data/interim/metadata/113_reviewed_donor_evidence.csv` with **exact columns** `case_uuid,g3_class,evidence_id,evidence_url,applicability,reviewed`. One row per affected candidate; `reviewed` must be `true`; `applicability` must identify the patient or an externally verified original cohort and its covered donor IDs. This file is a **manual adjudication input**, not a notebook-generated assurance. Its existence does not mean the protocol is formally approved.

Unlinked or undocumented cases stay `UNRESOLVED_INDEPENDENCE`. To protect against indiscriminate cohort-wide assignments, a shared evidence identifier requires case-specific applicability or separately verifiable cohort membership. The notebook validates fields and provenance but cannot independently prove documentary truth; that requires review.

In [4]:
evidence_sources={}
if G3_EVIDENCE.exists():
    ev=load(G3_EVIDENCE)
    required_ev={"case_uuid","g3_class","evidence_id","evidence_url","applicability","reviewed"}
    if not required_ev.issubset(ev.columns): raise ValueError(f"G3 evidence missing {required_ev-set(ev.columns)}")
    exact_unique(ev,["case_uuid"],"G3 evidence")
    if not ev.case_uuid.isin(ledger.case_uuid).all(): raise ValueError("G3 evidence includes outside-G2 patients")
    allowed={"DOCUMENTED_INDEPENDENT","DOCUMENTED_OVERLAP","UNRESOLVED_INDEPENDENCE"}
    if not ev.g3_class.isin(allowed).all(): raise ValueError("Invalid donor class")
    for c in ["evidence_id","evidence_url","applicability"]:
        if ev[c].isna().any() or ev[c].str.strip().eq("").any(): raise ValueError(f"G3 evidence lacks {c}")
    if not ev.reviewed.str.casefold().eq("true").all(): raise ValueError("G3 evidence not reviewed")
    ev=ev.set_index("case_uuid")
    for k in ["g3_class","evidence_id","applicability"]:
        col={"g3_class":"g3_class","evidence_id":"g3_evidence_id","applicability":"g3_reason"}[k]
        ledger[col]=ledger.case_uuid.map(ev[k]).fillna(ledger[col])
    ledger["g3_reviewed"]=ledger.case_uuid.isin(ev.index)
    evidence_sources["g3"]={"path":str(G3_EVIDENCE.relative_to(ROOT)),"sha256":sha(G3_EVIDENCE),"n_rows":len(ev)}
else:
    print("G3 reviewed evidence absent: retaining 1,082 UNRESOLVED statuses")
assert len(ledger)==1082 and ledger.case_uuid.is_unique

G3 reviewed evidence absent: retaining 1,082 UNRESOLVED statuses


## G4 / G5 / G6 / G8 — conservative admission gates

Raw GDC site/disease text is **not** a validated TCGA lineage. A G4 crosswalk, if reviewed, has exact keys `project_id,primary_site,disease_type,tcga_lineage,mapping_status,reviewed,evidence_id`; `mapping_status=ACCEPTED` alone authorizes mapping after review, otherwise remain unmapped. Optional file QC review requires `file_id,qc_status,reviewed,method_evidence_id`, with `qc_status` in `QC_PASS,QC_FAIL,QC_UNASSESSED`. QC PASS must not be inferred from provider MD5 or gene coverage.

G6 assay-level normalization and G8 inference-dependent confounder feasibility remain unresolved until their prospective criteria and data are available. No automatic technical eligibility or cohort freeze.

In [5]:
source_keys=["project_id","primary_site","disease_type"]
ledger["tcga_lineage"]=pd.NA
ledger["g4_status"]="UNMAPPED"
ledger["g4_evidence_id"]=pd.NA
if G4_CROSSWALK.exists():
    cross=load(G4_CROSSWALK)
    req=set(source_keys+["tcga_lineage","mapping_status","reviewed","evidence_id"])
    if not req.issubset(cross.columns): raise ValueError(f"G4 crosswalk missing {req-set(cross.columns)}")
    exact_unique(cross,source_keys,"G4 crosswalk")
    if not cross.reviewed.str.casefold().eq("true").all(): raise ValueError("Unreviewed G4 crosswalk")
    if not cross.mapping_status.isin(["ACCEPTED","UNMAPPED","AMBIGUOUS","CONTRADICTORY"]).all(): raise ValueError("Invalid G4 status")
    accepted=cross.mapping_status.eq("ACCEPTED")
    if cross.loc[accepted,["tcga_lineage","evidence_id"]].isna().any().any(): raise ValueError("G4 accepted source without lineage/evidence")
    ledger=ledger.merge(cross[source_keys+["tcga_lineage","mapping_status","evidence_id"]].rename(columns={"tcga_lineage":"g4_mapped","mapping_status":"g4_reviewed_status","evidence_id":"g4_reviewed_evidence"}),on=source_keys,how="left",validate="many_to_one")
    ledger["tcga_lineage"]=ledger["g4_mapped"]
    ledger["g4_status"]=ledger["g4_reviewed_status"].fillna("UNMAPPED")
    ledger["g4_evidence_id"]=ledger["g4_reviewed_evidence"]
    ledger=ledger.drop(columns=["g4_mapped","g4_reviewed_status","g4_reviewed_evidence"])
    evidence_sources["g4"]={"path":str(G4_CROSSWALK.relative_to(ROOT)),"sha256":sha(G4_CROSSWALK),"n_rows":len(cross)}
ledger["qc_status"]="QC_UNASSESSED"
ledger["qc_evidence_id"]=pd.NA
if QC_EVIDENCE.exists():
    qc=load(QC_EVIDENCE)
    req={"file_id","qc_status","reviewed","method_evidence_id"}
    if not req.issubset(qc.columns): raise ValueError(f"QC evidence missing {req-set(qc.columns)}")
    exact_unique(qc,["file_id"],"QC review")
    if not qc.file_id.isin(ledger.file_id).all(): raise ValueError("QC reviewed file outside G2")
    if not qc.reviewed.str.casefold().eq("true").all(): raise ValueError("Unreviewed QC status")
    if not qc.qc_status.isin(["QC_PASS","QC_FAIL","QC_UNASSESSED"]).all(): raise ValueError("Invalid QC status")
    if qc.method_evidence_id.isna().any(): raise ValueError("QC methods evidence ID missing")
    qc=qc.set_index("file_id")
    ledger["qc_status"]=ledger.file_id.map(qc.qc_status).fillna(ledger.qc_status)
    ledger["qc_evidence_id"]=ledger.file_id.map(qc.method_evidence_id)
    evidence_sources["qc"]={"path":str(QC_EVIDENCE.relative_to(ROOT)),"sha256":sha(QC_EVIDENCE),"n_rows":len(qc)}
ledger["g6_status"]="INPUT_COMPATIBILITY_UNASSESSED"
ledger["g8_status"]="CONFOUNDER_FEASIBILITY_UNASSESSED"
ledger["cohort_status"]="NOT_FROZEN"
ledger["exclusion_or_blocker"]=ledger.apply(lambda r: ";".join([x for x,fail in [
 ("G3_UNRESOLVED",r.g3_class=="UNRESOLVED_INDEPENDENCE"),
 ("G3_OVERLAP",r.g3_class=="DOCUMENTED_OVERLAP"),
 ("G4_UNMAPPED",r.g4_status!="ACCEPTED"),
 ("G5_QC_UNASSESSED",r.qc_status=="QC_UNASSESSED"),
 ("G5_QC_FAIL",r.qc_status=="QC_FAIL"),
 ("G6_UNASSESSED",True),("G8_UNASSESSED",True)
] if fail]),axis=1)
assert ledger.case_uuid.is_unique and len(ledger)==1082
display(ledger.groupby(["g3_class","g4_status","qc_status"],dropna=False).size().rename("n_cases").reset_index())

,g3_class,g4_status,qc_status,n_cases
0,UNRESOLVED_INDEPENDENCE,UNMAPPED,QC_UNASSESSED,1082


In [6]:
## Emit deterministic interim handoff. Freeze is intentionally disabled by this first implementation.
outputs={
"113_donor_identity_and_cohort_links.csv":ledger[["case_uuid","case_submitter_id","file_id","sample_uuid","aliquot_uuid","uuid_collision_flag","submitter_collision_flag","g3_class","g3_evidence_id","g3_reason","g3_reviewed"]].copy(),
"113_primary_candidate_case_eligibility.csv":ledger.copy(),
"113_source_to_tcga_lineage_crosswalk.csv":ledger[source_keys+["tcga_lineage","g4_status","g4_evidence_id"]].drop_duplicates().sort_values(source_keys),
"113_cohort_attrition_by_context.csv":ledger.groupby(source_keys,dropna=False).agg(g2_simple_cases=("case_uuid","nunique"),g2_primary_files=("file_id","nunique"),n_flagged_ambiguous=("other_ambiguous_rna_flag","sum"),n_documented_independent=("g3_class",lambda x:(x=="DOCUMENTED_INDEPENDENT").sum()),n_unresolved_independence=("g3_class",lambda x:(x=="UNRESOLVED_INDEPENDENCE").sum()),n_documented_overlap=("g3_class",lambda x:(x=="DOCUMENTED_OVERLAP").sum())).reset_index(),
"113_assay_qc_and_feature_compatibility.csv":ledger[["case_uuid","file_id","workflow_type","provider_md5","qc_status","qc_evidence_id","g6_status","g8_status"]].copy()
}
fprints={}
for name,df in outputs.items():
    path=OUT/name
    df.to_csv(path,index=False)
    fprints[name]={"sha256":sha(path),"n_rows":len(df)}
inputs={str(path.relative_to(ROOT)):sha(path) for path in [INDEX,PROVISIONAL,TCGA,REGISTRY]}
manifest={"notebook":113,"input_sha256":inputs,"reviewed_evidence":evidence_sources,"outputs":fprints,"source_counts":{"rna_files":2968,"methylation_files":2610,"provisional_primary_cases":1370,"provisional_primary_rna_files":1731,"g2_simple_cases":1082,"multiplicity_cases":288,"multiplicity_primary_files":649,"g2_additional_ambiguous_rna_flagged":37},"cohort_frozen":False}
manifest_path=OUT/"113_source_provenance_manifest.json"
manifest_path.write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\n",encoding="utf-8")
status={"notebook":113,"g2_policy":"APPROVED","g3_protocol":"DRAFT","g3_class_counts":ledger.g3_class.value_counts().to_dict(),"g4_status_counts":ledger.g4_status.value_counts().to_dict(),"qc_status_counts":ledger.qc_status.value_counts().to_dict(),"g6":"NOT_ADJUDICATED","g8":"NOT_ADJUDICATED","cohort_frozen":False,"external_analysis_authorized":False,"readiness":"BLOCKED","no_biological_analysis":True,"manifest_sha256":sha(manifest_path)}
status_path=OUT/"113_cohort_readiness.json"
status_path.write_text(json.dumps(status,indent=2,sort_keys=True)+"\n",encoding="utf-8")
display(pd.DataFrame([{"artifact":k,**v} for k,v in fprints.items()]))
print("Readiness:",status["readiness"],"; frozen:",status["cohort_frozen"],"; output:",status_path.relative_to(ROOT))

,artifact,sha256,n_rows
0,113_donor_identity_and_cohort_links.csv,6b5f01076c694e7aefe695b0f848588ffe8f90bb7945c1...,1082
1,113_primary_candidate_case_eligibility.csv,2e91b2584f648bbd83ed82a175a9193720ddf466e893ee...,1082
2,113_source_to_tcga_lineage_crosswalk.csv,3a752b68eb29eef44d7ff54a02467f1dd6858f99b1ba53...,18
3,113_cohort_attrition_by_context.csv,81b902a0714e4f6710affde92074e3dfafc1188d1990f1...,18
4,113_assay_qc_and_feature_compatibility.csv,18666d2677cd0e4f8f0f5038de73783be4dbaee769920f...,1082


Readiness: BLOCKED ; frozen: False ; output: data\interim\metadata\113_cohort_readiness.json


## Interpretation and next decision

- The output is a **G2-exact evidence ledger**, not a validated independent external cohort.
- G3 documentary adjudications require an **approved** protocol, verified patient-to-original-cohort mapping and reviewed evidence; zero exact collisions are not decisive.
- G4 and QC review inputs are deliberately optional: missing evidence yields *unresolved*, not an invented PASS.
- To finalize a cohort in a later reviewed execution, specify and validate G6 normalization/input compatibility, G8 confounder feasibility, lineage-aware inferential support and G7 interface with 800 **before** inspecting frozen program scores. A frozen membership list and registry update require a separate explicit go/no-go.
- No new data retrieval, program scoring, statistical replication or retrospective selection occurs here.